# S&P 500 动量因子选股

**目标**: 用纯量价动量特征 + LightGBM 在 S&P 500 中预测未来 10 天残差收益率，选出 top 30 股票等权持有。

**Pipeline**: 数据下载 → 特征工程（~50 个量价特征）→ 标签构造（10 天残差收益）→ LightGBM + Purged CV → 回测 → 分析

**已知偏差**:
- 使用当前 S&P 500 成分股，存在 **survivorship bias**（存活者偏差）
- 残差收益用 `r_stock - r_SPY` 近似，理论上应用 beta-adjusted 残差 `r_i - β_i × r_m`

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from equity.data import get_sp500_tickers, download_stock_data, load_universe, load_spy
from equity.features import compute_all_features, cross_sectional_rank
from equity.labels import make_labels
from equity.model import run_cv_pipeline
from equity.backtest import backtest_topN, compute_daily_returns
from equity.analysis import (
    summary_metrics, print_metrics,
    calc_ic_series, quantile_analysis,
    plot_equity_curve, plot_ic_analysis, plot_quantile_returns, plot_feature_importance,
)

pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. 数据下载

下载 S&P 500 全部成分股 + SPY 的日线 OHLCV（2015-2026）。首次运行需要几分钟，之后从 parquet 缓存加载。

In [ ]:
tickers = get_sp500_tickers()
print(f"S&P 500 tickers: {len(tickers)}")
print(f"Sample: {tickers[:10]}")

download_stock_data(tickers, start="2015-01-01", end="2026-04-01", cache_dir="../data/equity")

In [ ]:
# 加载数据
panel = load_universe(cache_dir="../data/equity", min_history_days=252)
spy = load_spy(cache_dir="../data/equity")

n_stocks = panel.index.get_level_values("ticker").nunique()
date_range = panel.index.get_level_values("date")
print(f"Universe: {n_stocks} stocks")
print(f"Date range: {date_range.min().date()} to {date_range.max().date()}")
print(f"Total rows: {len(panel):,}")
print(f"\nSPY: {len(spy)} days")
print(f"\nSample:")
panel.head()

## 2. 特征工程

计算 ~50 个量价动量特征，然后转截面百分位排名。

In [ ]:
%%time
raw_features = compute_all_features(panel)
print(f"Raw features shape: {raw_features.shape}")
print(f"Feature columns ({len(raw_features.columns)}):")
print(raw_features.columns.tolist())

In [ ]:
# 截面排名转换
features = cross_sectional_rank(raw_features)
print(f"Ranked features shape: {features.shape}")
print(f"Value range: [{features.min().min():.3f}, {features.max().max():.3f}]")
print(f"\nNaN ratio per feature:")
print((features.isna().sum() / len(features)).describe())

## 3. 标签构造

未来 10 天残差收益率 = 个股收益 - SPY 收益。

> **近似说明**: 理论上应用 β-adjusted 残差 (r_i - β_i × r_m)，这里假设所有股票 β=1。对于 β 远离 1 的股票（如公用事业 ~0.5、科技 ~1.3），标签会有系统性偏差。

In [ ]:
labels = make_labels(panel, spy, periods=10)
print(f"Labels shape: {labels.shape}")
print(f"NaN ratio: {labels.isna().mean():.3f}")
print(f"\nLabel distribution:")
labels.dropna().describe()

## 4. 单特征 IC 预检

在训练模型之前，先看每个特征单独的预测力（Spearman IC），筛掉完全无效的特征。

In [ ]:
from scipy import stats

# Align features and labels
common_idx = features.index.intersection(labels.dropna().index)
feat_aligned = features.loc[common_idx]
lab_aligned = labels.loc[common_idx]

# Compute mean IC for each feature
feature_ics = {}
for col in feat_aligned.columns:
    valid = feat_aligned[col].notna() & lab_aligned.notna()
    if valid.sum() < 1000:
        continue
    
    # Daily cross-sectional IC
    ic_daily = feat_aligned.loc[valid, [col]].copy()
    ic_daily["label"] = lab_aligned.loc[valid]
    
    daily_ic = ic_daily.groupby(level="date").apply(
        lambda g: stats.spearmanr(g[col], g["label"])[0] if len(g) > 10 else np.nan
    )
    feature_ics[col] = {
        "mean_ic": daily_ic.mean(),
        "ic_std": daily_ic.std(),
        "icir": daily_ic.mean() / daily_ic.std() if daily_ic.std() > 0 else 0,
        "pct_positive": (daily_ic > 0).mean(),
    }

ic_df = pd.DataFrame(feature_ics).T.sort_values("mean_ic", ascending=False)
print(f"Feature IC summary ({len(ic_df)} features):\n")
print(ic_df.to_string(float_format="{:.4f}".format))

## 5. 模型训练 — LightGBM + Purged CV

使用 Purged Time-Series CV 防止时间泄漏：
- 训练窗口：500 天（~2 年）
- 测试窗口：60 天（~3 个月）
- Purge gap：10 天（= 标签前瞻期）
- Embargo：5 天（额外缓冲）

In [ ]:
%%time
predictions, models = run_cv_pipeline(
    features=features,
    labels=labels,
    n_splits=5,
    train_days=500,
    test_days=60,
    purge_days=10,
    embargo_days=5,
)
print(f"\nOOS predictions: {len(predictions):,} samples")
print(f"Date range: {predictions.index.get_level_values('date').min().date()} "
      f"to {predictions.index.get_level_values('date').max().date()}")
predictions.head()

## 6. IC 分析

Information Coefficient = 每日预测值与实际收益的 Spearman 相关。
- IC > 0.02 说明有微弱信号
- ICIR > 0.5 说明信号稳定

In [ ]:
ic_series = calc_ic_series(predictions)
print(f"Mean IC:  {ic_series.mean():.4f}")
print(f"IC Std:   {ic_series.std():.4f}")
print(f"ICIR:     {ic_series.mean() / ic_series.std():.4f}")
print(f"IC > 0:   {(ic_series > 0).mean():.1%}")

plot_ic_analysis(ic_series)
plt.savefig("../output/equity_ic_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. 分层回测（Quantile Analysis）

按预测值分 5 组，如果模型有效，Q5（最高预测值）应该收益最高，Q1 最低，呈单调递增。

In [ ]:
q_returns = quantile_analysis(predictions, n_groups=5)
print("Annualized mean return by quintile:")
print((q_returns.mean() * 252).to_string(float_format="{:.2%}".format))
print(f"\nQ5 - Q1 spread: {(q_returns[5].mean() - q_returns[1].mean()) * 252:.2%} annualized")

plot_quantile_returns(q_returns)
plt.savefig("../output/equity_quantile_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. 回测 — Top 30 Long Only vs SPY

In [ ]:
# Compute daily returns for backtest (raw returns, not residual)
daily_returns = compute_daily_returns(panel)

# Run backtest
bt_results = backtest_topN(
    predictions=predictions,
    returns=daily_returns,
    top_n=30,
    rebalance_days=10,
    cost_bps=10,
)

# Strategy metrics
print("=== Strategy (Top 30, 10-day rebalance, 10bps cost) ===")
strat_metrics = summary_metrics(bt_results["portfolio_return"])
print_metrics(strat_metrics)

# SPY benchmark metrics (same period)
spy_daily_ret = spy["close"].pct_change().reindex(bt_results.index).fillna(0)
print("\n=== SPY Benchmark ===")
spy_metrics = summary_metrics(spy_daily_ret)
print_metrics(spy_metrics)

print(f"\n=== Alpha ===")
print(f"  Annual excess return: {strat_metrics['annual_return'] - spy_metrics['annual_return']:.2%}")
print(f"  Average turnover per rebalance: {bt_results['turnover'][bt_results['turnover'] > 0].mean():.1%}")

In [ ]:
# Equity curve
plot_equity_curve(bt_results, spy_daily_ret, title="Momentum Top-30 vs SPY")
plt.savefig("../output/equity_curve.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Feature Importance

In [ ]:
feature_names = features.columns.tolist()
plot_feature_importance(models, feature_names, top_n=20)
plt.savefig("../output/equity_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. 总结

**判断标准**:
- Mean IC > 0.02 且 ICIR > 0.5 → 信号有意义
- 分层回测 Q5 > Q4 > ... > Q1 → 模型有区分力
- 策略年化收益 > SPY 且扣费后为正 → pipeline 可用

**下一步改进方向**:
- 用 beta-adjusted 残差替代简单差值
- 加入历史成分股数据消除 survivorship bias
- 特征筛选（去掉 IC 接近 0 的特征）
- 调参（Optuna / 网格搜索）
- 加入财务因子（估值、盈利质量等）